# 02 - Methods: Baseline, Shaping, Curriculum, Imitation

**Goal:** Make the project method choices explicit before running long jobs.

**What you will learn:** What we are building, which methods are required, and which extras are optional.

**Inputs:** The project package and optional smoke-training artifacts.

**Outputs:** Method tables, config summaries, curriculum task table, and guarded optional small runs.

**Success criteria:** You can identify the required PPO baseline, reward shaping, and curriculum path.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

## Required And Optional Methods

The core path is three PPO agents. Optional methods are useful if time allows, but they are not the first thing to run.

In [ ]:
import pandas as pd

methods = pd.DataFrame([
    {"stage": "ppo_baseline", "role": "required", "purpose": "plain PPO comparison point"},
    {"stage": "ppo_shaped", "role": "required", "purpose": "reward-modification experiment"},
    {"stage": "ppo_curriculum", "role": "required", "purpose": "main performance candidate"},
    {"stage": "ppo_selfplay", "role": "optional", "purpose": "fallback if curriculum is weak"},
    {"stage": "bc_imitation", "role": "optional", "purpose": "imitate a stronger expert"},
    {"stage": "dqn_baseline", "role": "optional", "purpose": "algorithm comparison only"},
])
display(methods)

## PPO Baseline

Baseline PPO trains one controlled player against a simple opponent policy. It is the curve everything else compares against.

In [ ]:
try:
    from soccer_twos_project.training import STAGES, build_training_config
    from soccer_twos_project.config import select_profile
    profile = select_profile("auto", smoke=True)
    baseline_config = build_training_config("ppo_baseline", profile)
    print_json({"stage": STAGES["ppo_baseline"], "model": baseline_config.get("model"), "env_config": baseline_config.get("env_config")})
except Exception as exc:
    print("Baseline config inspection skipped:", type(exc).__name__, exc)

## Reward-Shaped PPO

Shaping adds small training rewards for progress toward useful soccer behavior. It should be compared directly against the baseline curve.

In [ ]:
try:
    shaped_config = build_training_config("ppo_shaped", profile)
    print_json(shaped_config["env_config"].get("reward_shaping"))
except Exception as exc:
    print("Shaping config inspection skipped:", type(exc).__name__, exc)

## Curriculum PPO

Curriculum starts with easier ball/player placements and progresses toward harder scenarios when performance improves.

In [ ]:
try:
    from soccer_twos_project.training import load_curriculum
    curriculum_rows = []
    for idx, task in enumerate(load_curriculum()):
        curriculum_rows.append({"task": idx, "name": task["name"], "config_fn": task["config_fn"]})
    display(pd.DataFrame(curriculum_rows))
except Exception as exc:
    print("Curriculum inspection skipped:", type(exc).__name__, exc)

## Optional Small Method Run

Use this only when you want to verify that a method writes logs before committing to the full pipeline.

In [ ]:
RUN_METHOD_SMOKE = True
METHOD_SMOKE_STAGE = "ppo_shaped"  # Try ppo_shaped or ppo_curriculum.
METHOD_SMOKE_TIMESTEPS = 5_000

if RUN_METHOD_SMOKE:
    run_training(ctx, METHOD_SMOKE_STAGE, profile_name="auto", timesteps=METHOD_SMOKE_TIMESTEPS, smoke=True, verbose=1)
else:
    print("Optional method smoke run skipped. Set RUN_METHOD_SMOKE=True to run one short method check.")

## Optional Extras

These are guarded by booleans. Leave them off until the required PPO path is working.

In [ ]:
RUN_SELFPLAY_SMOKE = True
RUN_DQN_SMOKE = True
RUN_IMITATION_EXAMPLE = True

if RUN_SELFPLAY_SMOKE:
    run_training(ctx, "ppo_selfplay", profile_name="auto", timesteps=5_000, smoke=True, verbose=1)
else:
    print("Self-play smoke skipped.")

if RUN_DQN_SMOKE:
    run_training(ctx, "dqn_baseline", profile_name="auto", timesteps=5_000, smoke=True, verbose=1)
else:
    print("DQN smoke skipped.")

if RUN_IMITATION_EXAMPLE:
    print("Run imitation after exporting a strong expert package such as soccer_ppo_curriculum.")
else:
    print("Imitation example skipped until an expert package exists.")

## Key Takeaways

The required solution path is baseline PPO, reward-shaped PPO, and curriculum PPO. Self-play, imitation, and DQN are optional extensions, not the first path to a valid submission.

## What To Run Next

Run `03_full_training_pipeline.ipynb` for the long training, export, and quick-evaluation workflow.